In [15]:
# ============================================================
# 🦾 MASTER DASHBOARD: Research Edition (Dynamics + Kinematics)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D 
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob
import os
import re

# 1. Setup & Geometry (Updated for Rev 5.2)
BASE_LOG_DIR = os.path.join("..", "..", "logs")

# Hardware Geometry
L_BASE = 10.0   # cm (Updated)
L1 = 20.0       # cm (Updated)
L2 = 20.0       # cm (Updated)
MAX_ALLOWED_ERROR = 0.5 # cm

# 2. Define the Main Analysis Logic
def load_and_analyze(run_folder_path):
    clear_output(wait=True)
    
    run_name = os.path.basename(run_folder_path)
    is_3d = False
    has_dynamics = False
    
    # --- DETECT LOG TYPE ---
    log_dyn = glob.glob(os.path.join(run_folder_path, "arm_dynamics_log.csv"))
    log_3d = glob.glob(os.path.join(run_folder_path, "arm_3d_log.csv"))
    log_2d = glob.glob(os.path.join(run_folder_path, "arm_2link_log.csv"))
    
    if log_dyn:
        df = pd.read_csv(log_dyn[0])
        is_3d = True
        has_dynamics = True
        print(f"📂 Loaded RESEARCH DATA (Dynamics): {run_name}")
    elif log_3d:
        df = pd.read_csv(log_3d[0])
        is_3d = True
        print(f"📂 Loaded Standard 3D Data: {run_name}")
    elif log_2d:
        df = pd.read_csv(log_2d[0])
        print(f"📂 Loaded 2D Data: {run_name}")
    else:
        print(f"❌ No data found in {run_name}")
        return

    # Normalize Time
    if 'timestamp_ms' in df.columns:
        df['time_sec'] = df['timestamp_ms'] / 1000.0
    else:
        df['time_sec'] = np.arange(len(df)) * 0.002 # Fallback
    
    # --- PRE-CALCULATE GEOMETRY (JOINT POSITIONS) ---
    try:
        if is_3d:
            rad_base = np.radians(df['base_deg'])
            rad_shld = np.radians(df['shoulder_deg'])
            
            # Elbow Position Calculation
            r_elbw = L1 * np.cos(rad_shld)
            df['elbow_z'] = L_BASE + L1 * np.sin(rad_shld)
            df['elbow_x'] = r_elbw * np.cos(rad_base)
            df['elbow_y'] = r_elbw * np.sin(rad_base)
        else:
            # Fallback for 2D logs
            rad_m1 = np.radians(df['motor1_pos']) if 'motor1_pos' in df else np.zeros(len(df))
            df['elbow_x'] = L1 * np.cos(rad_m1)
            df['elbow_y'] = L1 * np.sin(rad_m1)
            df['elbow_z'] = 0 
            df['base_deg'] = 0 
            df['shoulder_deg'] = 0
            
    except Exception as e:
        print(f"⚠️ Warning: Geometry calculation issue: {e}")

    # --- METRICS ---
    duration = df['time_sec'].max()
    max_error = df['error'].max() if 'error' in df else 0.0
    avg_error = df['error'].mean() if 'error' in df else 0.0
    verdict = "PASS" if max_error <= MAX_ALLOWED_ERROR else "FAIL"
    
    # ========================================================
    # 📊 PART 1: STATIC ANALYSIS REPORT (v6.0 - Error Focus)
    # ========================================================
    try:
        # UPDATED LAYOUT: Wider and Higher
        fig_static = plt.figure(figsize=(16, 12)) # Made bigger overall
        
        # New Grid: Left column is all ONE big plot (3D), Right column is split 3 ways
        gs = fig_static.add_gridspec(3, 2, width_ratios=[1.5, 1]) 
        
        # --- Plot 1: 3D Path (ENTIRE LEFT COLUMN) ---
        # Spanning all 3 rows of column 0 -> gs[:, 0]
        if is_3d:
            ax1 = fig_static.add_subplot(gs[:, 0], projection='3d')
            ax1.set_title(f"1. 3D Spatial Path (Sag Visualized)", fontweight='bold')
            ax1.plot(df['target_x'], df['target_y'], df['target_z'], 'g--', label='Target', alpha=0.5)
            ax1.plot(df['real_x'], df['real_y'], df['real_z'], 'b-', alpha=0.8, label='Actual (Sag)', lw=2)
            ax1.set_zlabel("Z (cm)")
            last = df.iloc[-1]
            ax1.plot([0, 0, last['elbow_x'], last['real_x']], 
                     [0, 0, last['elbow_y'], last['real_y']], 
                     [0, L_BASE, last['elbow_z'], last['real_z']], 'r-o', lw=3)
            ax1.legend()
            # Force equal aspect ratio so it doesn't look squashed
            ax1.set_box_aspect([1,1,1]) 
        else:
            ax1 = fig_static.add_subplot(gs[:, 0])
            ax1.plot(df['target_x'], df['target_y'], 'g--')
            ax1.plot(df['real_x'], df['real_y'], 'b-')

        ax1.set_xlabel("X (cm)")
        ax1.set_ylabel("Y (cm)")

        # --- Plot 2: COMBINED TORQUE (Split View) ---
        # Split the top-right cell into 2 stacked plots
        gs_torque = gs[0, 1].subgridspec(2, 1, hspace=0.3)
        
        if has_dynamics:
            # Limits
            lim_s = df['limit_shoulder'].iloc[0]
            lim_e = df['limit_elbow'].iloc[0]
            lim_b = df['limit_base'].iloc[0] if 'limit_base' in df else 0.45 
            
            # --- SUBPLOT A: SHOULDER (High Torque) ---
            ax2_top = fig_static.add_subplot(gs_torque[0, 0])
            ax2_top.set_title("2a. Shoulder Torque (High Load)", fontweight='bold', fontsize=10)
            ax2_top.plot(df['time_sec'], df['torque_shoulder'], 'r-', lw=1.5, label='Shoulder')
            ax2_top.axhline(lim_s, color='r', linestyle=':', alpha=0.5)
            ax2_top.axhline(-lim_s, color='r', linestyle=':', alpha=0.5)
            ax2_top.set_ylabel("Nm")
            ax2_top.grid(True, alpha=0.3)
            # Remove x-ticks labels to save space if needed, but keeping them is safer
            
            # --- SUBPLOT B: BASE & ELBOW (Low Torque) ---
            ax2_bot = fig_static.add_subplot(gs_torque[1, 0])
            ax2_bot.set_title("2b. Base & Elbow (Low Load)", fontweight='bold', fontsize=10)
            if 'torque_base' in df:
                ax2_bot.plot(df['time_sec'], df['torque_base'], 'g-', lw=1.5, label='Base')
            ax2_bot.plot(df['time_sec'], df['torque_elbow'], 'b-', lw=1.5, label='Elbow')
            
            ax2_bot.axhline(lim_e, color='b', linestyle=':', alpha=0.5)
            ax2_bot.axhline(-lim_e, color='b', linestyle=':', alpha=0.5)
            ax2_bot.set_ylabel("Nm")
            ax2_bot.legend(loc='upper right', fontsize='small')
            ax2_bot.grid(True, alpha=0.3)
            
        else:
            ax2 = fig_static.add_subplot(gs[0, 1])
            ax2.text(0.3, 0.5, "No Dynamics Data")

        # --- Plot 3: SPATIAL ERROR (Middle Right) ---
        ax3 = fig_static.add_subplot(gs[1, 1])
        ax3.set_title("3. Spatial Error vs Time", fontweight='bold', color='purple')
        
        if 'error' in df:
            ax3.plot(df['time_sec'], df['error'], color='purple', lw=2, label='Euclidean Error')
            ax3.axhline(MAX_ALLOWED_ERROR, color='k', linestyle='--', lw=2)
            
            ax3.fill_between(df['time_sec'], df['error'], MAX_ALLOWED_ERROR, 
                             where=(df['error'] > MAX_ALLOWED_ERROR), color='red', alpha=0.3)
            
            ax3.set_ylabel("Error (cm)")
            ax3.grid(True, alpha=0.3)
        else:
            ax3.text(0.3, 0.5, "No Error Data")

        # --- Plot 4: Summary Metrics (Bottom Right) ---
        ax_text = fig_static.add_subplot(gs[2, 1])
        ax_text.axis('off')
        
        # Calculate RMS
        rms_error = np.sqrt((df['error']**2).mean()) if 'error' in df else 0.0

        dyn_text = ""
        if has_dynamics:
            max_s = df['torque_shoulder'].abs().max()
            max_e = df['torque_elbow'].abs().max()
            max_b = df['torque_base'].abs().max() if 'torque_base' in df else 0.0
            
            lim_s = df['limit_shoulder'].iloc[0]
            lim_e = df['limit_elbow'].iloc[0]
            lim_b = df['limit_base'].iloc[0] if 'limit_base' in df else 0.45
            
            util_s = (max_s / lim_s) * 100
            util_e = (max_e / lim_e) * 100
            util_b = (max_b / lim_b) * 100 if lim_b > 0 else 0
            
            dyn_text = f"""
        MOTOR HEALTH:
        -------------------------
        Base Peak:     {max_b:.2f} Nm ({util_b:.0f}%)
        Shoulder Peak: {max_s:.1f} Nm ({util_s:.0f}%)
        Elbow Peak:    {max_e:.1f} Nm ({util_e:.0f}%)
            """
        else:
            dyn_text = "\n        (No Dynamic Data)\n"
        
        verdict_color = "FAIL" if max_error > MAX_ALLOWED_ERROR else "PASS"
        
        summary_text = f"""
        RUN ANALYSIS: {run_name}
        -----------------------------
        Duration:   {duration:.2f} s
        
        ACCURACY (SAG IMPACT):
        ----------------
        Max Error:  {max_error:.4f} cm
        Avg Error:  {avg_error:.4f} cm
        RMS Error:  {rms_error:.4f} cm
        Result:     {verdict_color}
        {dyn_text}
        """
        
        ax_text.text(0.05, 0.5, summary_text, fontsize=11, family='monospace', va='center')
        
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Plot Error: {e}")
        import traceback
        traceback.print_exc()

    # ========================================================
    # 🎮 PART 2: 3D DIGITAL TWIN
    # ========================================================
    print("\n👇 3D DIGITAL TWIN (Drag Slider to Replay) 👇")
    
    def plot_3d_frame(frame_index, elev, azim):
        try:
            fig_3d = plt.figure(figsize=(10, 8))
            ax = fig_3d.add_subplot(111, projection='3d')
            
            if frame_index >= len(df): frame_index = len(df) - 1
            row = df.iloc[frame_index]
            
            # --- COLOR LOGIC (Stress Detection) ---
            shldr_color = 'orange'
            elbw_color = 'orange'
            s_warn = ""
            e_warn = ""
            
            if has_dynamics:
                if abs(row['torque_shoulder']) > (row['limit_shoulder'] * 0.9): 
                    shldr_color = 'red'; s_warn = "(!)"
                if abs(row['torque_elbow']) > (row['limit_elbow'] * 0.9): 
                    elbw_color = 'red'; e_warn = "(!)"

            # --- SETUP PLOT ---
            ax.set_xlim(-25, 35)
            ax.set_ylim(-25, 35)
            ax.set_zlim(0, 40)
            
            # --- INFO TITLE ---
            info = f"t={row['time_sec']:.2f}s | Err: {row['error']:.2f}cm"
            if has_dynamics:
                info += f"\nShoulder: {row['torque_shoulder']:.1f}Nm {s_warn} | Elbow: {row['torque_elbow']:.1f}Nm {e_warn}"
            ax.set_title(info, fontsize=11, fontweight='bold')
            
            # --- GEOMETRY ---
            ax.plot([-25, 35, 35, -25, -25], [-25, -25, 35, 35, -25], [0,0,0,0,0], 'k-', alpha=0.1) 
            
            # Trace
            hist = df.iloc[:frame_index+1:15] 
            ax.plot(df['target_x'], df['target_y'], df['target_z'], 'g--', alpha=0.1)
            ax.plot(hist['real_x'], hist['real_y'], hist['real_z'], 'b-', alpha=0.4)

            # Robot
            if is_3d:
                ax.plot([0, 0], [0, 0], [0, L_BASE], 'k-', lw=4) # Base
                ax.plot([0, row['elbow_x']], [0, row['elbow_y']], [L_BASE, row['elbow_z']], 
                        color=shldr_color, lw=6, solid_capstyle='round') # Link 1
                ax.plot([row['elbow_x'], row['real_x']], [row['elbow_y'], row['real_y']], 
                        [row['elbow_z'], row['real_z']], 
                        color=elbw_color, lw=5, solid_capstyle='round') # Link 2
                ax.scatter([0, row['elbow_x'], row['real_x']], 
                           [0, row['elbow_y'], row['real_y']], 
                           [L_BASE, row['elbow_z'], row['real_z']], 
                           color='k', s=60, zorder=10) # Joints
            
            ax.view_init(elev=elev, azim=azim)
            plt.show()
            
        except Exception as e:
            print(f"Frame Error: {e}")

    avg_dt = df['time_sec'].diff().mean()
    if np.isnan(avg_dt) or avg_dt == 0: avg_dt = 0.002
    PLAY_STEP = 25 
    real_interval = int(avg_dt * PLAY_STEP * 1000)
    safe_interval = max(30, real_interval)

    play = widgets.Play(
        value=0, min=0, max=len(df)-1,
        step=PLAY_STEP, interval=safe_interval, description="Play"
    )
    slider_time = widgets.IntSlider(min=0, max=len(df)-1, step=PLAY_STEP, value=0, layout=widgets.Layout(width='60%'))
    widgets.jslink((play, 'value'), (slider_time, 'value'))
    
    slider_elev = widgets.IntSlider(min=0, max=90, step=5, value=25, description='Elev')
    slider_azim = widgets.IntSlider(min=0, max=360, step=5, value=300, description='Azim')
    
    ui = widgets.VBox([
        widgets.HBox([play, slider_time]), 
        widgets.HBox([slider_elev, slider_azim])
    ])
    
    out = widgets.interactive_output(plot_3d_frame, {
        'frame_index': slider_time, 
        'elev': slider_elev, 
        'azim': slider_azim
    })
    
    display(ui, out)

# 3. Main Selection Logic
all_folders = [f.path for f in os.scandir(BASE_LOG_DIR) if f.is_dir()]
arm_folders = sorted(
    [f for f in all_folders if "arm_run" in os.path.basename(f)],
    key=lambda x: int(re.search(r"arm_run_?(\d+)", os.path.basename(x)).group(1)) 
                  if re.search(r"arm_run_?(\d+)", os.path.basename(x)) else 0
)

dashboard_output = widgets.Output()

if not arm_folders:
    print("❌ No 'arm_run' folders found.")
else:
    # Default to the most recent run
    dropdown = widgets.Dropdown(
        options=[(os.path.basename(f), f) for f in arm_folders],
        value=arm_folders[-1],
        description='Select Run:',
        layout=widgets.Layout(width='50%')
    )
    
    def on_run_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with dashboard_output:
                load_and_analyze(change['new'])

    dropdown.observe(on_run_change)
    display(dropdown, dashboard_output)
    
    # Init Load
    with dashboard_output:
        load_and_analyze(arm_folders[-1])

Dropdown(description='Select Run:', index=1, layout=Layout(width='50%'), options=(('arm_run_3d_5', '..\\..\\lo…

Output()